# TATR / TMTR — Reduced Replication
## FTS-Diffusion Paper, Section 5.3 / Fig. 6

### Multi-asset, multi-experiment notebook
Supports: **S&P 500**, **GOOG**, **ZC=F** for both **TATR** and **TMTR** experiments.

### Configuration (reduced from paper to fit Colab Pro A100)
- **15 runs** (paper: 100) — for 95% CI estimate
- **101 augmentation steps × 252 days = 100 years** of synthetic data per run (matches paper)
- **Train LSTM at 11 evaluation points**: 0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100 years
- **Total LSTMs trained**: 15 × 11 = **165** per asset (paper: ~10,100)

### Persistent storage on Drive
```
/content/drive/MyDrive/fts_diffusion/
├── architectures/{asset}/   # Pre-trained FTS-Diffusion per asset
├── synthetic/{asset}/       # Generated synthetic series per run
├── results/{exp}/{asset}/   # MAPE CSVs + summary
└── figures/{exp}/           # Final plots
```

### Paper-faithful hyperparameters
| Param | Value |
|-------|-------|
| LSTM hidden_dim | 32 |
| LSTM layers | 2 |
| Window size | 64 |
| Loss | MAE |
| Optimizer | Adam, lr=1e-2 |
| Epochs | 100 |
| Training mode | **Full-batch** |

## 1. Environment Setup

In [ ]:
import os, sys, subprocess, time, json, shutil
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print(f"Running on Colab: {IS_COLAB}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_URL = 'https://github.com/thaidinger/ml-in-fcs.git'
    BRANCH = 'deli_work'
    CLONE_DIR = '/content/ml-in-fcs'
    if not os.path.exists(CLONE_DIR):
        subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, CLONE_DIR])
    else:
        subprocess.check_call(['git', '-C', CLONE_DIR, 'pull'])

    REPO_ROOT = f'{CLONE_DIR}/fts-diffusion-ref'
    PERSIST_ROOT = '/content/drive/MyDrive/fts_diffusion'
else:
    REPO_ROOT = '/home/deli/ETH_projects/ml-in-fcs/fts-diffusion-ref'
    PERSIST_ROOT = '/home/deli/ETH_projects/ml-in-fcs/fts_diffusion_run'

# Create top-level Drive structure (subdirs created lazily once ASSET/EXPERIMENT chosen)
os.makedirs(PERSIST_ROOT, exist_ok=True)
for sub in ['architectures', 'synthetic', 'results', 'figures']:
    os.makedirs(f'{PERSIST_ROOT}/{sub}', exist_ok=True)

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
for d in ['data', 'res', 'trained_models', 'figs']:
    os.makedirs(d, exist_ok=True)

print(f"Working dir:    {os.getcwd()}")
print(f"Persistent dir: {PERSIST_ROOT}")

In [ ]:
# Install dependencies
for pkg in ['numpy', 'pandas', 'matplotlib', 'scipy', 'scikit-learn', 'torch', 'tqdm', 'tslearn', 'dtaidistance', 'yfinance']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("\u2713 Deps ready")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

# GPU info
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # For maximum reproducibility (slightly slower):
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("\u2713 Deterministic mode ON (TF32 disabled)")

# Authors' modules
from utils.load_data import get_fts, load_actual_fts
from models.model_params import prm_params, pgm_params, pem_params
from models.train_ftsdiffusion import (
    train_ftsdiffusion,
    _has_recognition_artifacts, _has_generation_artifact, _has_evolution_artifact)
from models.sampling import generate_timeseries_ftsdiffusion
from experiments.utils_downstream import (
    setup_dowmstream_tatr, init_first_segment,
    Timeseries2Dataset_Downstream, create_syn_dataset,
    concat_datasets_downstream, copy_dataset_downstream)
from experiments.predictor_lstm import LSTMPredictor, set_loss_fn

print("\u2713 Imports ready")

## 2. Configuration

In [ ]:
# ============================================================================
# === EXPERIMENT CONFIG ===
# Change ASSET and EXPERIMENT to switch dataset / experiment type.
# All paths on Drive are derived from these two variables.
# ============================================================================

ASSET          = 'sp500'      # 'sp500' | 'goog' | 'zcf'
EXPERIMENT     = 'tatr'       # 'tatr'  | 'tmtr'

# Per-asset metadata (matches paper Appendix E.1)
ASSETS_CONFIG = {
    'sp500': {'ticker': '^GSPC', 'start': '1980-01-01', 'end': '2020-01-01', 'pretty': 'S&P 500'},
    'goog':  {'ticker': 'GOOG',  'start': '2005-01-01', 'end': '2020-01-01', 'pretty': 'GOOG'},
    'zcf':   {'ticker': 'ZC=F',  'start': '2001-01-01', 'end': '2020-01-01', 'pretty': 'ZC=F (Corn futures)'},
}
assert ASSET in ASSETS_CONFIG, f"Unknown asset {ASSET}"
assert EXPERIMENT in ['tatr', 'tmtr'], f"Unknown experiment {EXPERIMENT}"

DATANAME       = ASSET
TICKER         = ASSETS_CONFIG[ASSET]['ticker']
START_DATE     = ASSETS_CONFIG[ASSET]['start']
END_DATE       = ASSETS_CONFIG[ASSET]['end']
PRETTY_NAME    = ASSETS_CONFIG[ASSET]['pretty']

# === Persistent paths derived from ASSET / EXPERIMENT ===
ARCH_DIR    = f'{PERSIST_ROOT}/architectures/{ASSET}'
SYN_DIR     = f'{PERSIST_ROOT}/synthetic/{ASSET}'
RES_DIR     = f'{PERSIST_ROOT}/results/{EXPERIMENT}/{ASSET}'
FIG_DIR     = f'{PERSIST_ROOT}/figures/{EXPERIMENT}'
for d in [ARCH_DIR, SYN_DIR, RES_DIR, FIG_DIR,
          f'{ARCH_DIR}/res', f'{ARCH_DIR}/trained_models', f'{ARCH_DIR}/data']:
    os.makedirs(d, exist_ok=True)

# === Override the global prm_params with the chosen asset ===
from models.model_params import prm_params
prm_params['dataname'] = ASSET

# === Experiment hyperparameters ===
N_RUNS         = 15

# EVAL_YEARS: which augmentation lengths to evaluate the LSTM at.
# This list is MUTABLE: you can add/remove years between sessions.
# - Already-computed years are loaded from disk (no recomputation).
# - New years are computed using the EXISTING saved synthetic data.
# - As long as max(EVAL_YEARS) <= SYN_MAX_YEARS, no synthetic regeneration is needed.
EVAL_YEARS     = [0, 10, 30, 50, 70, 90]

# SYN_MAX_YEARS: how much synthetic data to generate ONCE per run (independent of EVAL_YEARS).
# Make this generous (e.g. 100) so you can add new EVAL_YEARS in the future
# without regenerating synthetic data.
# WARNING: if you increase this AFTER having already saved synthetic .npy files
# with a smaller length, the runner will REGENERATE them automatically.
SYN_MAX_YEARS  = 100

AUG_LENGTH     = 252

# Sanity: every EVAL_YEAR must be \u2264 SYN_MAX_YEARS
assert max(EVAL_YEARS) <= SYN_MAX_YEARS, (
    f"max(EVAL_YEARS)={max(EVAL_YEARS)} > SYN_MAX_YEARS={SYN_MAX_YEARS}. "
    f"Either reduce EVAL_YEARS or increase SYN_MAX_YEARS.")

# === LSTM CONFIG (paper-faithful) ===
WINDOW_SIZE    = 64
LSTM_HIDDEN    = 32
LSTM_LAYERS    = 2
AHEAD          = 1
N_EPOCHS       = 100
LR             = 1e-2
LSTM_LOSS      = 'mae'
DATATYPE       = 'prices'

print(f"Configuration:")
print(f"  ASSET         = {ASSET}  ({PRETTY_NAME})")
print(f"  EXPERIMENT    = {EXPERIMENT.upper()}")
print(f"  N_RUNS        = {N_RUNS}")
print(f"  EVAL_YEARS    = {EVAL_YEARS}  (mutable; can be extended later)")
print(f"  SYN_MAX_YEARS = {SYN_MAX_YEARS}  (synthetic generated once per run)")
print(f"  Total LSTMs   = {N_RUNS * len(EVAL_YEARS)} (this session)")
print()
print(f"Drive paths:")
print(f"  ARCH_DIR = {ARCH_DIR}")
print(f"  SYN_DIR  = {SYN_DIR}")
print(f"  RES_DIR  = {RES_DIR}")
print(f"  FIG_DIR  = {FIG_DIR}")

## 3. Phase 1 — Train FTS-Diffusion Architecture (Once)

This step is **expensive (~1 hour on A100)** but only runs once. Checkpoints persist on Drive.

In [ ]:
# Sync persistent checkpoints to working dirs (so train_ftsdiffusion finds them)

def _is_for_asset(filename, asset, kind):
    """Return True if filename belongs to the given asset."""
    if kind == 'res':
        # SISC artifacts: sisc_{asset}_k{K}_l{lmin}-{lmax}_*.csv
        return filename.startswith(f'sisc_{asset}_') and filename.endswith('.csv')
    if kind == 'trained_models':
        # PGM/PEM filenames contain the asset name (e.g. pgm-2_c48-80_sp500_*)
        return (f'_{asset}_' in filename) and (filename.endswith('.pth') or filename.endswith('.pt'))
    if kind == 'data':
        return filename == f'{asset}_timeseries.csv'
    return False


def sync_persistent_to_working(persistent_dir, working_subdir, asset=None, kind=None):
    """Copy from Drive into working dir if working dir is empty (per file).
    Optionally filter by asset/kind to avoid cross-asset contamination."""
    src = persistent_dir
    dst = os.path.join(REPO_ROOT, working_subdir)
    if not os.path.exists(src):
        return
    for f in os.listdir(src):
        src_f = os.path.join(src, f)
        dst_f = os.path.join(dst, f)
        if not os.path.isfile(src_f):
            continue
        if asset is not None and kind is not None and not _is_for_asset(f, asset, kind):
            continue
        if not os.path.exists(dst_f):
            shutil.copy(src_f, dst_f)
            print(f"  Restored {f}")


def sync_working_to_persistent(working_subdir, persistent_dir, asset=None, kind=None):
    """Copy from working dir to Drive (overwrite). Filter by asset/kind for clean separation."""
    src = os.path.join(REPO_ROOT, working_subdir)
    dst = persistent_dir
    os.makedirs(dst, exist_ok=True)
    if not os.path.exists(src):
        return
    for f in os.listdir(src):
        src_f = os.path.join(src, f)
        dst_f = os.path.join(dst, f)
        if not os.path.isfile(src_f):
            continue
        if asset is not None and kind is not None and not _is_for_asset(f, asset, kind):
            continue
        shutil.copy(src_f, dst_f)


# Restore architecture checkpoints if they exist on Drive (filtered by current ASSET)
print(f"Checking for existing {ASSET} architecture checkpoints on Drive...")
sync_persistent_to_working(f'{ARCH_DIR}/res',             'res',             asset=ASSET, kind='res')
sync_persistent_to_working(f'{ARCH_DIR}/trained_models',  'trained_models',  asset=ASSET, kind='trained_models')
sync_persistent_to_working(f'{ARCH_DIR}/data',            'data',            asset=ASSET, kind='data')

In [ ]:
# Step 1: Download asset data if not already present
ts_path = f'data/{DATANAME}_timeseries.csv'
if not os.path.exists(ts_path):
    print(f"Downloading {TICKER} from {START_DATE} to {END_DATE}...")
    get_fts(ticker=TICKER, fts_name=DATANAME, start_date=START_DATE, end_date=END_DATE)
else:
    print(f"\u2713 Data already at {ts_path}")

fts = load_actual_fts(DATANAME).squeeze()
print(f"{PRETTY_NAME} series: {len(fts)} points, range [{fts.min():.4f}, {fts.max():.4f}]")

In [ ]:
# Step 2: Train SISC + PGM + PEM (only if no checkpoint for this asset)
has_all = (_has_recognition_artifacts() and
           _has_generation_artifact() and
           _has_evolution_artifact())

if has_all:
    print(f"\u2713 All {ASSET} architecture artifacts found \u2014 skipping training.")
else:
    print(f"Training FTS-Diffusion architecture for {PRETTY_NAME} (SISC + PGM + PEM)...")
    print("Estimated time: ~50-75 min on A100")
    t0 = time.time()
    train_ftsdiffusion(fts, train_test_split=0.8, store_model=True)
    elapsed = (time.time() - t0) / 60
    print(f"\u2713 Architecture training complete in {elapsed:.1f} min")

# Persist asset-specific checkpoints to Drive
print(f"Syncing {ASSET} checkpoints to Drive...")
sync_working_to_persistent('res', f'{ARCH_DIR}/res', asset=ASSET, kind='res')
sync_working_to_persistent('trained_models', f'{ARCH_DIR}/trained_models', asset=ASSET, kind='trained_models')
sync_working_to_persistent('data', f'{ARCH_DIR}/data', asset=ASSET, kind='data')
print(f"\u2713 {ASSET} checkpoints persisted on Drive at {ARCH_DIR}.")

## 4. Phase 2 — TATR Loop

For each of the 15 runs:
1. Generate `MAX_YEARS × 252 = 25,200` days of synthetic data (~12 min on A100)
2. For each evaluation year in `[0, 10, 20, ..., 100]`:
   - Build augmented dataset = real init + synthetic [0:year × 252]
   - Train LSTM (full-batch, 100 epochs, lr=1e-2, MAE loss, hidden=32)
   - Test on real test set → record MAPE
3. Save run results to CSV after each run.

In [ ]:
# === Setup TATR datasets (ADAPTIVE: handles short datasets like GOOG/ZC=F) ===
#
# The authors' setup_dowmstream_tatr() hardcodes init_period = 252*5 days (5 years),
# which corresponds to ~62.5% of the test set length for S&P 500 (5y init / 8y test).
# For shorter datasets (GOOG, ZC=F) the hardcoded value exceeds the available test data
# and the code crashes.
#
# To stay paper-faithful, we keep the same init/test RATIO as S&P 500 in the paper:
#     init_fraction = 5 / 8 = 0.625
# Then the absolute init period scales with each asset's available test data.
# This keeps S&P 500 identical to the paper (1260 days / 756 days) and provides a
# principled, documented choice for GOOG and ZC=F.

from experiments.utils_downstream import (
    get_downstream_data, init_first_segment, init_tatr_set,
    Timeseries2Dataset_Downstream
)


# Per-asset init/test split. Default = 0.625 (= 5/8, same as paper S&P 500).
# Override here if you want a different ratio for a specific asset.
INIT_FRACTION_PER_ASSET = {
    'sp500': 0.625,   # = 5/8: matches paper hardcoding (5y init / 3y test)
    'goog':  0.625,   # same ratio: ~1.9y init / ~1.1y test
    'zcf':   0.625,   # same ratio: ~2.5y init / ~1.5y test
}


def setup_dowmstream_tatr_adaptive(window_size, init_fraction=0.625, min_test_years=1):
    """Adaptive TATR setup that uses init_fraction of the test data as initial LSTM training set.

    Args:
        window_size:    LSTM input window length
        init_fraction:  fraction of available test data to use as LSTM init (default 0.625, paper-like)
        min_test_years: enforce at least this many years for actual LSTM testing
    """
    downstream_timeseries, segments_test, labels_test, lengths_test = get_downstream_data()
    total_len = len(downstream_timeseries)

    init_period = int(total_len * init_fraction)
    # Safety: leave at least min_test_years for testing, and at least 1y for init
    max_init = total_len - 252 * min_test_years
    init_period = min(init_period, max_init)
    init_period = max(init_period, 252)

    init_timeseries = downstream_timeseries[:init_period]
    test_timeseries = downstream_timeseries[init_period:]

    init_dataset, scaler = init_tatr_set(init_timeseries, window_size)
    first_segment = init_first_segment(segments_test, labels_test, lengths_test)
    test_dataset = init_tatr_set(test_timeseries, window_size, scaler)

    print(f"Adaptive TATR setup for {ASSET} ({PRETTY_NAME}):")
    print(f"  Total downstream timeseries: {total_len} days ({total_len/252:.2f} years)")
    print(f"  init_fraction:               {init_fraction:.3f}")
    print(f"  Init period (LSTM training): {init_period} days ({init_period/252:.2f} years)")
    print(f"  Test period (LSTM eval):     {total_len - init_period} days ({(total_len-init_period)/252:.2f} years)")
    print(f"  init_dataset shape:  {init_dataset.shape}")
    print(f"  test_dataset shape:  {test_dataset.shape}")
    return init_dataset, first_segment, test_dataset, scaler


# Pick the init fraction for the current ASSET (defaults to paper-like 0.625)
init_fraction = INIT_FRACTION_PER_ASSET.get(ASSET, 0.625)
init_dataset, first_segment, test_dataset, scaler = setup_dowmstream_tatr_adaptive(
    WINDOW_SIZE, init_fraction=init_fraction)
init_state, init_segment = first_segment

In [ ]:
# === LSTM training & evaluation functions (paper-faithful) ===

def train_lstm_full_batch(dataset, n_epochs=N_EPOCHS, hidden_dim=LSTM_HIDDEN,
                          n_layers=LSTM_LAYERS, output_dim=AHEAD, criterion_str=LSTM_LOSS,
                          lr=LR, seed=0):
    """Match the authors' full-batch training in predictor_lstm.py exactly."""
    torch.manual_seed(seed)
    model = LSTMPredictor(input_dim=1, hidden_dim=hidden_dim, output_dim=output_dim,
                          n_layers=n_layers, device=device).to(device)
    criterion = set_loss_fn(criterion_str)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    X = dataset[:, :-output_dim].unsqueeze(-1).to(device)
    y = dataset[:, -output_dim:].to(device)
    
    for epoch in range(n_epochs):
        model.train()
        y_pred = model(X)
        loss = criterion(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    return model, float(loss.item())


def evaluate_lstm(model, test_dataset, scaler, output_dim=AHEAD):
    """Compute MAPE on the real test set, on the original (un-scaled) scale."""
    model.eval()
    with torch.no_grad():
        X = test_dataset[:, :-output_dim].unsqueeze(-1).to(device)
        y_true = test_dataset[:, -output_dim:].numpy()
        y_pred = model(X).cpu().numpy()
        # Inverse-transform back to original scale
        y_true_orig = scaler.inverse_transform(y_true)
        y_pred_orig = scaler.inverse_transform(y_pred)
    return float(MAPE(y_true_orig, y_pred_orig))

print("\u2713 Train/eval functions ready")

In [ ]:
# === TATR run for a single seed (with INCREMENTAL per-year checkpointing) ===

def run_single_tatr(run_idx, eval_years, seed_base=42):
    """Run one TATR replicate with incremental per-year saving.

    Resumability:
      - At start, loads any years already saved in {RES_DIR}/run_XX.csv
      - For each year in eval_years, skips if already in CSV (no re-train)
      - After each NEW year is computed, immediately saves to CSV

    Synthetic data:
      - Generated ONCE per run, with length SYN_MAX_YEARS * AUG_LENGTH
      - If a saved file is shorter than needed, AUTOMATICALLY regenerated
      - Adding new EVAL_YEARS (within SYN_MAX_YEARS) requires no regeneration

    Adding new evaluation years later:
      - Append more years to EVAL_YEARS in the config cell
      - This function will load all previously-saved years (skip them)
      - Compute only the new ones, append to the same CSV
    """
    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'

    # === Load any pre-existing (possibly partial) results ===
    results = {}
    if os.path.exists(run_csv):
        df_prev = pd.read_csv(run_csv)
        results = dict(zip(df_prev['eval_year'].astype(int), df_prev['mape'].astype(float)))
        if set(eval_years).issubset(set(results.keys())):
            print(f"  \u2713 Run {run_idx}: all requested years already in CSV \u2014 skipping.")
            return {ey: results[ey] for ey in eval_years}
        elif results:
            done = sorted(results.keys())
            missing = sorted(set(eval_years) - set(done))
            print(f"  \u26a0 Run {run_idx} partial: years already saved={done}, to compute={missing}")

    np.random.seed(seed_base + run_idx)
    torch.manual_seed(seed_base + run_idx)

    # === Synthetic data: ensure it covers SYN_MAX_YEARS ===
    syn_path = f'{SYN_DIR}/run_{run_idx:02d}_syn.npy'
    needed_length = SYN_MAX_YEARS * AUG_LENGTH
    full_syn = None

    if os.path.exists(syn_path):
        existing = np.load(syn_path)
        existing_years = len(existing) // AUG_LENGTH
        if existing_years >= SYN_MAX_YEARS:
            print(f"  Loading saved synthetic data for run {run_idx} ({existing_years} years available)")
            full_syn = existing
        else:
            print(f"  \u26a0 Saved synthetic covers only {existing_years} years "
                  f"(< SYN_MAX_YEARS={SYN_MAX_YEARS}). Regenerating...")
            # Regenerate (we lose the previous .npy, but the saved MAPE results stay)
            t0 = time.time()
            full_syn = generate_timeseries_ftsdiffusion(
                T=needed_length,
                init_state=init_state,
                init_segment=init_segment)
            np.save(syn_path, full_syn)
            print(f"  Synthetic regeneration: {(time.time()-t0)/60:.1f} min \u2014 saved to {syn_path}")

    if full_syn is None:
        print(f"  Generating {SYN_MAX_YEARS} years of synthetic data (~{SYN_MAX_YEARS//8} min on A100)...")
        t0 = time.time()
        full_syn = generate_timeseries_ftsdiffusion(
            T=needed_length,
            init_state=init_state,
            init_segment=init_segment)
        np.save(syn_path, full_syn)
        print(f"  Synthetic generation: {(time.time()-t0)/60:.1f} min \u2014 saved to {syn_path}")

    # === Per-year evaluation with INCREMENTAL save ===
    for ey in eval_years:
        if ey in results:
            print(f"  Run {run_idx:2d} | Year {ey:3d} | already done \u2014 MAPE={results[ey]:.5f} (loaded)")
            continue

        if ey * AUG_LENGTH > len(full_syn):
            # Should never happen now (we regenerate above), but safety net
            print(f"  Run {run_idx:2d} | Year {ey:3d} | SKIPPED (synthetic too short)")
            continue

        if ey == 0:
            curr_dataset = copy_dataset_downstream(init_dataset)
        else:
            syn_chunk = full_syn[:ey * AUG_LENGTH]
            syn_dataset = create_syn_dataset(syn_chunk, WINDOW_SIZE, scaler, DATATYPE)
            curr_dataset = concat_datasets_downstream(
                copy_dataset_downstream(init_dataset), syn_dataset)

        t0 = time.time()
        model, train_loss = train_lstm_full_batch(curr_dataset, seed=seed_base + run_idx + ey)
        train_time = time.time() - t0
        mape = evaluate_lstm(model, test_dataset, scaler)
        results[ey] = mape

        print(f"  Run {run_idx:2d} | Year {ey:3d} | windows={curr_dataset.shape[0]:6d} | "
              f"train_loss={train_loss:.4f} | MAPE={mape:.5f} | LSTM={train_time:.1f}s")

        # === SAVE TO DRIVE IMMEDIATELY (the critical fix) ===
        sorted_years = sorted(results.keys())
        df_save = pd.DataFrame({
            'eval_year': sorted_years,
            'mape': [results[y] for y in sorted_years]
        })
        df_save.to_csv(run_csv, index=False)

        # Free GPU memory
        del model, curr_dataset
        torch.cuda.empty_cache()

    return {ey: results[ey] for ey in eval_years if ey in results}

print("\u2713 TATR runner ready (SYN_MAX_YEARS-aware, auto-regen, incremental save)")

In [ ]:
# ============================================================================
# === MAIN LOOP \u2014 runs all replicates with live plotting + mean tracking ===
# ============================================================================
%matplotlib inline
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

all_results = {}
running_mape = np.full((N_RUNS, len(EVAL_YEARS)), np.nan)

# === STEP 1: Pre-load any runs already saved on Drive ===
print(f"Pre-loading existing runs from {RES_DIR}...")
for run_idx in range(N_RUNS):
    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if os.path.exists(run_csv):
        df = pd.read_csv(run_csv)
        for j, ey in enumerate(EVAL_YEARS):
            mask = df['eval_year'] == ey
            if mask.any():
                running_mape[run_idx, j] = df.loc[mask, 'mape'].values[0]
        all_results[run_idx] = dict(zip(df['eval_year'], df['mape']))
n_preloaded = int(sum(~np.all(np.isnan(running_mape), axis=1)))
print(f"  \u2713 Found {n_preloaded} existing runs on Drive\n")


# === STEP 2: Mean tracker ===
mean_history = []

def update_mean_history():
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    if n_done == 0:
        return None
    means = np.nanmean(running_mape[valid], axis=0)
    snap = {'n_runs': n_done}
    for j, ey in enumerate(EVAL_YEARS):
        snap[f'mean_y{ey}'] = float(means[j]) if not np.isnan(means[j]) else None
    mean_history.append(snap)
    return means


# === STEP 3: Live plot helper ===
def plot_live(elapsed_min=None):
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    if n_done == 0:
        return

    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(12, 7))

    for r in range(N_RUNS):
        if not valid[r]:
            continue
        # Skip NaN points (years not yet computed for this run)
        mask = ~np.isnan(running_mape[r])
        ax.plot(np.array(EVAL_YEARS)[mask], running_mape[r][mask],
                color='steelblue', alpha=0.3, linewidth=1)

    means = np.nanmean(running_mape[valid], axis=0) if n_done >= 1 else np.full(len(EVAL_YEARS), np.nan)
    if n_done >= 2:
        stds = np.nanstd(running_mape[valid], axis=0)
        ax.fill_between(EVAL_YEARS, means - stds, means + stds,
                        alpha=0.25, color='steelblue', label=f'\u00b11 std (n={n_done})')
    valid_mean = ~np.isnan(means)
    ax.plot(np.array(EVAL_YEARS)[valid_mean], means[valid_mean],
            'o-', linewidth=2.5, color='darkblue', markersize=8, label='Mean MAPE')
    if not np.isnan(means[0]):
        ax.axhline(means[0], color='gray', linestyle='--', alpha=0.7,
                   label=f'Year-0 baseline = {means[0]:.4f}')

    for x, y in zip(EVAL_YEARS, means):
        if np.isnan(y):
            continue
        ax.annotate(f'{y:.4f}', xy=(x, y), xytext=(0, 12),
                    textcoords='offset points', ha='center',
                    fontsize=9, fontweight='bold', color='darkblue',
                    bbox=dict(boxstyle='round,pad=0.3', fc='white',
                              ec='darkblue', alpha=0.85, linewidth=0.8))

    title = f'{EXPERIMENT.upper()} live \u2014 {PRETTY_NAME} \u2014 {n_done}/{N_RUNS} runs done'
    if elapsed_min is not None:
        title += f' (last: {elapsed_min:.1f} min)'
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Augmentation (years)', fontsize=11)
    ax.set_ylabel('MAPE on real test set', fontsize=11)
    ax.legend(loc='best', fontsize=10)
    ax.grid(alpha=0.3)
    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax + 0.015)
    plt.tight_layout()
    display(fig)
    plt.close(fig)


# === STEP 4: Mean delta printer ===
def print_mean_delta(prev_means, new_means, n_done):
    if prev_means is None or new_means is None:
        if new_means is not None:
            print(f"\n[Mean tracking] First run completed (n=1). Initial means:")
            for ey, m in zip(EVAL_YEARS, new_means):
                if not np.isnan(m):
                    print(f"  Year {ey:3d}: {m:.5f}")
        return
    print(f"\n[Mean tracking] After run {n_done} (n={n_done}):")
    print(f"  {'Year':>4} | {'Old mean':>10} | {'New mean':>10} | {'\u0394':>10}")
    print(f"  {'-'*4}-+-{'-'*10}-+-{'-'*10}-+-{'-'*10}")
    for ey, old, new in zip(EVAL_YEARS, prev_means, new_means):
        if np.isnan(old) and np.isnan(new):
            print(f"  {ey:>4d} | {'NaN':>10} | {'NaN':>10} | {'\u2014':>10}")
            continue
        if np.isnan(old):
            print(f"  {ey:>4d} | {'NaN':>10} | {new:>10.5f} | NEW")
            continue
        if np.isnan(new):
            print(f"  {ey:>4d} | {old:>10.5f} | {'NaN':>10} | {'\u2014':>10}")
            continue
        delta = new - old
        sign = '\u2193' if delta < 0 else ('\u2191' if delta > 0 else '=')
        print(f"  {ey:>4d} | {old:>10.5f} | {new:>10.5f} | {sign} {abs(delta):.5f}")


# === STEP 5: Initialize history with pre-loaded data ===
prev_means = update_mean_history()
plot_live()


# === STEP 6: Main run loop ===
for run_idx in range(N_RUNS):
    print(f"\n{'='*70}")
    print(f"Run {run_idx + 1}/{N_RUNS} \u2014 {PRETTY_NAME} ({EXPERIMENT.upper()})")
    print(f"{'='*70}")

    t0 = time.time()
    results = run_single_tatr(run_idx, EVAL_YEARS)
    all_results[run_idx] = results
    elapsed = (time.time() - t0) / 60

    # === FIX: handle missing years gracefully (no KeyError) ===
    for j, ey in enumerate(EVAL_YEARS):
        running_mape[run_idx, j] = results.get(ey, np.nan)

    new_means = update_mean_history()
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    print_mean_delta(prev_means, new_means, n_done)
    prev_means = new_means.copy() if new_means is not None else None

    plot_live(elapsed_min=elapsed)
    pd.DataFrame(mean_history).to_csv(f'{RES_DIR}/mean_history.csv', index=False)

    print(f"\u2713 Run {run_idx + 1} done in {elapsed:.1f} min")

print(f"\n{'='*70}\nALL {N_RUNS} RUNS COMPLETE for {PRETTY_NAME} ({EXPERIMENT.upper()})\n{'='*70}")

## 5. Phase 3 — Aggregation & Confidence Intervals

In [ ]:
# Reload all per-run CSVs from Drive (handles re-runs, partial completion)
rows = []
for run_idx in range(N_RUNS):
    p = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if not os.path.exists(p):
        print(f"\u26a0 Missing run {run_idx}")
        continue
    df = pd.read_csv(p)
    df['run_idx'] = run_idx
    rows.append(df)

df_all = pd.concat(rows, ignore_index=True)
df_pivot = df_all.pivot(index='run_idx', columns='eval_year', values='mape')
print(f"Results matrix: {df_pivot.shape} (runs \u00d7 eval_years)")
print("\nMAPE per run \u00d7 evaluation year:")
print(df_pivot.round(5))

In [ ]:
# Compute statistics: mean, 95% CI via percentile bootstrap
from scipy import stats as sstats

def bootstrap_ci(x, n_boot=10000, ci=0.95, seed=0):
    """Percentile bootstrap CI for the mean."""
    rng = np.random.RandomState(seed)
    n = len(x)
    boots = np.array([rng.choice(x, size=n, replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.quantile(boots, [(1-ci)/2, 1-(1-ci)/2])
    return lo, hi

summary_stats = []
for ey in EVAL_YEARS:
    vals = df_pivot[ey].dropna().values
    mean = vals.mean()
    std = vals.std()
    lo, hi = bootstrap_ci(vals)
    summary_stats.append({
        'eval_year': ey,
        'mean_mape': mean,
        'std_mape': std,
        'ci95_low': lo,
        'ci95_high': hi,
        'n_runs': len(vals)
    })

summary_df = pd.DataFrame(summary_stats)
print("\nSummary statistics (95% bootstrap CI over runs):")
print(summary_df.round(5))

# Save
summary_df.to_csv(f'{RES_DIR}/summary.csv', index=False)
df_pivot.to_csv(f'{RES_DIR}/results_matrix.csv')
print(f"\n\u2713 Saved to {RES_DIR}/summary.csv and results_matrix.csv")

## 6. Final Figure — Replicate Fig. 6(b) of the Paper

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

x = summary_df['eval_year'].values * AUG_LENGTH
mean = summary_df['mean_mape'].values
lo = summary_df['ci95_low'].values
hi = summary_df['ci95_high'].values

ax.plot(x, mean, 'o-', linewidth=2.5, color='steelblue', markersize=7,
        label=f'FTS-Diffusion (n={N_RUNS} runs)')
ax.fill_between(x, lo, hi, alpha=0.25, color='steelblue', label='95% bootstrap CI')
ax.axhline(mean[0], color='gray', linestyle='--', alpha=0.7,
           label=f'Initial MAPE (year 0) = {mean[0]:.4f}')

# Annotate mean on each marker
for xv, yv in zip(x, mean):
    ax.annotate(f'{yv:.4f}', xy=(xv, yv), xytext=(0, 12),
                textcoords='offset points', ha='center', fontsize=9,
                fontweight='bold', color='darkblue',
                bbox=dict(boxstyle='round,pad=0.3', fc='white',
                          ec='darkblue', alpha=0.85, linewidth=0.8))

if EVAL_YEARS[-1] == 100:
    pct_change = 100 * (mean[-1] - mean[0]) / mean[0]
    ax.annotate(f'\u0394MAPE @ year 100: {pct_change:+.1f}%\n(paper: -17.9%)',
                xy=(x[-1], mean[-1]), xytext=(x[-1]*0.55, mean[-1] + 0.02),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=11, color='red')

ax.set_xlabel('Augmented Size (days)', fontsize=12)
ax.set_ylabel('MAPE (real test set)', fontsize=12)
ax.set_title(f'{EXPERIMENT.upper()} \u2014 {PRETTY_NAME} \u2014 {N_RUNS} runs \u00d7 {len(EVAL_YEARS)} eval points '
             f'= {N_RUNS*len(EVAL_YEARS)} LSTMs', fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(alpha=0.3)
ymin, ymax = ax.get_ylim()
ax.set_ylim(ymin, ymax + 0.015)
plt.tight_layout()

fig_path = f'{FIG_DIR}/{EXPERIMENT}_{ASSET}.pdf'
plt.savefig(fig_path, bbox_inches='tight', dpi=150)
plt.savefig(fig_path.replace('.pdf', '.png'), bbox_inches='tight', dpi=150)
plt.show()
print(f"\u2713 Figure saved to {fig_path}")

## 7. Quality Checks

In [ ]:
# === Sanity checks ===
print("\n" + "="*70)
print("SANITY CHECKS")
print("="*70)

# 1) Every run completed?
print(f"\n[1] Runs completed: {len(df_pivot)}/{N_RUNS}")

# 2) Per-eval-year counts (should all be N_RUNS)
print(f"\n[2] Run counts per eval year:")
print(df_pivot.notna().sum(axis=0).to_string())

# 3) Trend: mean MAPE should generally DECREASE with more synthetic data (paper claim)
print(f"\n[3] Trend check — mean MAPE per eval year:")
print(summary_df[['eval_year', 'mean_mape']].to_string(index=False))
trend = np.polyfit(summary_df['eval_year'], summary_df['mean_mape'], 1)
print(f"  Linear trend slope: {trend[0]:.2e} ({'\u2193 downward (good)' if trend[0] < 0 else '\u2191 upward (NOT matching paper)'})")

# 4) Variance: across-run std should be reasonable (not all runs collapsing)
print(f"\n[4] Across-run std at each eval year:")
print(summary_df[['eval_year', 'std_mape']].to_string(index=False))

# 5) Year 0 baseline — paper reports ~0.026-0.05 MAPE
y0_mean = summary_df.iloc[0]['mean_mape']
print(f"\n[5] Year-0 baseline MAPE: {y0_mean:.5f} (paper Fig.6(b) starts ~0.05)")

# 6) Compute % reduction at year 100
if EVAL_YEARS[-1] == 100:
    y100_mean = summary_df.iloc[-1]['mean_mape']
    pct = 100 * (y0_mean - y100_mean) / y0_mean
    print(f"\n[6] MAPE reduction year 0 \u2192 100: {pct:+.2f}% (paper: 17.9%)")

## 8. How to Resume / Switch Asset / Add TMTR

### Resume after Colab disconnect
1. Reopen this notebook
2. Re-run cells 1-3 (setup, imports, config)
3. Re-run Phase 1 — it detects existing checkpoints for the current ASSET on Drive and skips
4. Re-run the main TATR loop — it detects completed runs in `RES_DIR` and skips them

### Switch to a different asset
1. Change `ASSET = 'goog'` (or `'zcf'`) in the **Configuration** cell
2. Re-run from Phase 1 onwards
3. Phase 1 will train the architecture for the new asset (~1h on A100)
4. Phase 2 will run the experiment for that asset
5. Results stored separately under `{PERSIST_ROOT}/results/{exp}/{ASSET}/`

### Switch from TATR to TMTR
1. Change `EXPERIMENT = 'tmtr'` in Configuration
2. (You will need a TMTR-specific main loop \u2014 currently this notebook implements TATR. The folder structure is ready for TMTR results to be stored at `{PERSIST_ROOT}/results/tmtr/{ASSET}/`.)

### Drive storage layout
```
/content/drive/MyDrive/fts_diffusion/
  \u251c\u2500\u2500 architectures/{asset}/   \u2190 SISC + PGM + PEM checkpoints (per asset)
  \u251c\u2500\u2500 synthetic/{asset}/        \u2190 Per-run synthetic series .npy
  \u251c\u2500\u2500 results/{exp}/{asset}/    \u2190 Per-run MAPE CSVs + summary + mean_history
  \u2514\u2500\u2500 figures/{exp}/            \u2190 Final plots
```

## Caveats / Sources of Variability

Even with `torch.manual_seed`, exact reproducibility on different GPUs is hard:
- **Different GPU = different cuDNN kernels** \u2192 small numerical differences
- **Sampling pipeline uses randomness in the diffusion noise** \u2192 different seed = different synthetic series
- **Across-run variance** is the *intended* signal: it captures uncertainty from generative randomness

If your year-100 MAPE reduction is in the range -10% to -25%, you've replicated the paper qualitatively.